In [ ]:
import shutil
import subprocess
import numpy as np
from pathlib import Path


def generate_pod_mode_visualizations(project_names, num_modes, inv_scale):
    """
    For each project name in project_names and each POD mode i:
      1. Copy {proj}_podMode_{i}.dat -> PODMode.dat
      2. Write the inversion factor to inversion_factor.txt
      3. Run the ParaView trace script Style1Final.py
      4. Copy ModeVisualization.png -> {proj}_podMode_{i}_Visualization.png

    Parameters
    ----------
    project_names : list of str
        Names of projects.
    num_modes : int
        Number of POD modes to visualize.
    inv_scale : array-like of shape (n_projects, num_modes)
        Inversion factors for each project and mode, typically 1 or -1.
    """

    # Absolute path to ParaView's pvpython executable
    pvpython = "/Applications/ParaView-5.11.1.app/Contents/bin/pvpython"

    # ParaView trace script
    pv_script = "Style1Final.py"

    # Working directory
    data_dir = Path(".")

    # Convert to numpy array for shape checking
    inv_scale = np.asarray(inv_scale)

    n_projects = len(project_names)

    if inv_scale.shape != (n_projects, num_modes):
        raise ValueError(
            f"inv_scale must have shape ({n_projects}, {num_modes}), "
            f"but got {inv_scale.shape}"
        )

    for p_idx, proj in enumerate(project_names):
        print(f"\nProcessing project: {proj}")

        for mode_idx in range(1, num_modes + 1):
            print(f"  Visualizing POD mode {mode_idx}")

            # Input file for this project/mode
            podmode_src = data_dir / f"{proj}_podMode_{mode_idx}.dat"

            # Temporary filename expected by ParaView script
            podmode_dst = data_dir / "PODMode.dat"

            # Temporary inversion factor file for ParaView script
            inversion_file = data_dir / "inversion_factor.txt"

            # Output image generated by Style1Final.py
            generated_img = data_dir / "ModeVisualization.png"

            # Final renamed image
            final_img = data_dir / f"{proj}_podMode_{mode_idx}_Visualization.png"

            # Check input file exists
            if not podmode_src.exists():
                raise FileNotFoundError(f"Missing file: {podmode_src}")

            # Copy project/mode-specific file to generic name expected by ParaView
            shutil.copy(podmode_src, podmode_dst)

            # Write inversion factor for this project/mode
            inversion_factor = inv_scale[p_idx, mode_idx - 1]
            with open(inversion_file, "w") as f:
                f.write(f"{inversion_factor}\n")

            # Run ParaView trace script
            subprocess.run([pvpython, pv_script], check=True)

            # Check that the script generated the expected output image
            if not generated_img.exists():
                raise FileNotFoundError(
                    f"Expected output image was not created: {generated_img}"
                )

            # Copy/rename output image
            shutil.copy(generated_img, final_img)

            print(f"  Saved visualization: {final_img}")

    print("\nAll POD mode visualizations generated.")